# Analise de Telemetria de Colheita

Este notebook organiza a analise operacional em 4 blocos:

1. Consumo por maquina por estado
2. Presenca diaria da frota (patio) e maquinas que nao trabalharam
3. Onde ficaram os registros sem apontamento
4. Picos horarios de paradas especificas (ex.: FALTA CAMINHAO)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)


In [ ]:
# Caminho da base consolidada (ajuste se necessario)
DATA_PATH = Path('telemetria_colheita_extraida/consolidado_telemetria_colheita.csv')
assert DATA_PATH.exists(), f'Arquivo nao encontrado: {DATA_PATH}'

# Lista do patio. Se quiser fixar manualmente, substitua pela sua lista.
PATIO_MAQUINAS = None  # ex.: ['5001', '5016', '5045', '5518', '5526']

# Parada alvo para analise de pico horario
PARADA_ALVO = 'FALTA CAMINHAO'


In [ ]:
df = pd.read_csv(DATA_PATH, sep=';', low_memory=False)

# Conversoes basicas
for col in ['vl_tempo_segundos', 'vl_consumo_instantaneo', 'vl_velocidade', 'vl_rpm']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['dt_hr_local_inicial'] = pd.to_datetime(df['dt_hr_local_inicial'], errors='coerce')
df = df.dropna(subset=['dt_hr_local_inicial']).copy()

# Campos derivados
df['data'] = df['dt_hr_local_inicial'].dt.date
df['hora'] = df['dt_hr_local_inicial'].dt.hour

df['cd_equipamento'] = df['cd_equipamento'].astype(str)
df['cd_estado'] = df['cd_estado'].astype(str).str.upper().str.strip()
df['desc_parada'] = df['desc_parada'].fillna('*NAO_DEFINIDO*').astype(str).str.strip()
df['desc_operador'] = df['desc_operador'].fillna('*NAO_DEFINIDO*').astype(str).str.strip()

# Mapa de estado de processo
estado_map = {
    'E': 'Efetivo',
    'P': 'Parada',
    'F': 'Parada',
    'M': 'Manobra',
    'D': 'Deslocamento',
}
df['estado_processo'] = df['cd_estado'].map(estado_map).fillna('Outros')

# Horas e consumo estimado
# Premissa: vl_consumo_instantaneo em L/h (ajuste se a origem for diferente)
df['horas_evento'] = df['vl_tempo_segundos'].fillna(0).clip(lower=0) / 3600.0
df['consumo_l_estimado'] = df['vl_consumo_instantaneo'].fillna(0).clip(lower=0) * df['horas_evento']

print('Linhas:', len(df))
print('Periodo:', df['dt_hr_local_inicial'].min(), '->', df['dt_hr_local_inicial'].max())
print('Maquinas:', df['cd_equipamento'].nunique())
print('Estados processo:', df['estado_processo'].value_counts().to_dict())


## 1) Consumo por maquina por estado

Leitura principal:
- `consumo_l_estimado`: litros estimados por evento (L/h * horas)
- `consumo_l_h`: intensidade media no estado (litros por hora no estado)


In [ ]:
consumo_maquina_estado = (
    df.groupby(['cd_equipamento', 'estado_processo'], as_index=False)
      .agg(
          horas=('horas_evento', 'sum'),
          consumo_l=('consumo_l_estimado', 'sum')
      )
)
consumo_maquina_estado['consumo_l_h'] = np.where(
    consumo_maquina_estado['horas'] > 0,
    consumo_maquina_estado['consumo_l'] / consumo_maquina_estado['horas'],
    np.nan
)

consumo_maquina_estado = consumo_maquina_estado.sort_values(['cd_equipamento', 'estado_processo'])
consumo_maquina_estado.head(20)


In [ ]:
fig = px.bar(
    consumo_maquina_estado,
    x='cd_equipamento',
    y='consumo_l',
    color='estado_processo',
    barmode='stack',
    title='Consumo estimado por maquina e estado (litros)',
    labels={'cd_equipamento': 'Maquina', 'consumo_l': 'Consumo estimado (L)', 'estado_processo': 'Estado'}
)
fig.update_layout(template='plotly_white', height=520)
fig.show()


In [ ]:
fig = px.bar(
    consumo_maquina_estado,
    x='cd_equipamento',
    y='consumo_l_h',
    color='estado_processo',
    barmode='group',
    title='Intensidade de consumo por maquina e estado (L/h)',
    labels={'cd_equipamento': 'Maquina', 'consumo_l_h': 'Consumo (L/h)', 'estado_processo': 'Estado'}
)
fig.update_layout(template='plotly_white', height=520)
fig.show()


## 2) Patio: quem trabalhou em cada dia e quem nao trabalhou

Definicao usada:
- `Trabalhou` no dia = teve `horas_efetivo > 0`
- Se teve registro mas `horas_efetivo == 0`, analisamos o motivo pelos estados/paradas do dia
- Se nao teve registro no dia, classificado como `Sem registro`


In [ ]:
if PATIO_MAQUINAS is None:
    patio = sorted(df['cd_equipamento'].dropna().astype(str).unique().tolist())
else:
    patio = sorted(pd.Series(PATIO_MAQUINAS).astype(str).unique().tolist())

print('Qtd maquinas no patio considerado:', len(patio))
print(patio)


In [ ]:
dias = sorted(df['data'].dropna().unique().tolist())
grade = pd.MultiIndex.from_product([dias, patio], names=['data', 'cd_equipamento']).to_frame(index=False)

resumo_dia_maquina = (
    df.groupby(['data', 'cd_equipamento'], as_index=False)
      .agg(
          horas_total=('horas_evento', 'sum'),
          horas_efetivo=('horas_evento', lambda s: s[df.loc[s.index, 'estado_processo'].eq('Efetivo')].sum()),
          horas_parada=('horas_evento', lambda s: s[df.loc[s.index, 'estado_processo'].eq('Parada')].sum()),
          eventos=('horas_evento', 'size')
      )
)

painel_patio = grade.merge(resumo_dia_maquina, on=['data', 'cd_equipamento'], how='left')
for c in ['horas_total', 'horas_efetivo', 'horas_parada', 'eventos']:
    painel_patio[c] = painel_patio[c].fillna(0)

painel_patio['status_trabalho'] = np.select(
    [painel_patio['eventos'].eq(0), painel_patio['horas_efetivo'].gt(0)],
    ['Sem registro', 'Trabalhou'],
    default='Sem efetivo'
)

painel_patio.head(20)


In [ ]:
presenca_diaria = (
    painel_patio.groupby('data', as_index=False)
      .agg(
          maquinas_patio=('cd_equipamento', 'nunique'),
          trabalharam=('status_trabalho', lambda s: (s == 'Trabalhou').sum()),
          sem_efetivo=('status_trabalho', lambda s: (s == 'Sem efetivo').sum()),
          sem_registro=('status_trabalho', lambda s: (s == 'Sem registro').sum())
      )
)

fig = go.Figure()
fig.add_bar(x=presenca_diaria['data'], y=presenca_diaria['trabalharam'], name='Trabalhou', marker_color='#2a9d8f')
fig.add_bar(x=presenca_diaria['data'], y=presenca_diaria['sem_efetivo'], name='Sem efetivo', marker_color='#e9c46a')
fig.add_bar(x=presenca_diaria['data'], y=presenca_diaria['sem_registro'], name='Sem registro', marker_color='#c1121f')
fig.update_layout(
    barmode='stack',
    template='plotly_white',
    height=420,
    title='Situacao diaria das maquinas do patio',
    xaxis_title='Data',
    yaxis_title='Qtd de maquinas'
)
fig.show()


In [ ]:
# Motivos para maquinas sem efetivo (teve registro, mas nao trabalhou em efetivo)
sem_efetivo_keys = painel_patio.loc[painel_patio['status_trabalho'].eq('Sem efetivo'), ['data', 'cd_equipamento']]
sem_efetivo = df.merge(sem_efetivo_keys, on=['data', 'cd_equipamento'], how='inner')

motivos_sem_efetivo = (
    sem_efetivo.groupby(['desc_parada'], as_index=False)
      .agg(horas=('horas_evento', 'sum'), eventos=('desc_parada', 'size'))
      .sort_values('horas', ascending=False)
)

motivos_sem_efetivo.head(20)


In [ ]:
fig = px.bar(
    motivos_sem_efetivo.head(15),
    x='horas',
    y='desc_parada',
    orientation='h',
    title='Top motivos (parada) quando maquina ficou sem efetivo',
    labels={'horas': 'Horas', 'desc_parada': 'Parada'}
)
fig.update_layout(template='plotly_white', height=500)
fig.show()


## 3) Onde ficou o sem apontamento

Regra inicial de sem apontamento:
- `desc_parada == *NAO_DEFINIDO*`
- ou `desc_operador == *NAO_DEFINIDO*`
- ou `cd_operador == -1`

Obs.: podemos endurecer/afrouxar a regra conforme seu padrao operacional.


In [ ]:
sem_apontamento = (
    df['desc_parada'].str.upper().eq('*NAO_DEFINIDO*')
    | df['desc_operador'].str.upper().eq('*NAO_DEFINIDO*')
    | df['cd_operador'].astype(str).str.strip().eq('-1')
)

df_sem = df[sem_apontamento].copy()
print('Registros sem apontamento:', len(df_sem), f"({len(df_sem)/len(df):.1%})")

res_sem_maquina = (
    df_sem.groupby('cd_equipamento', as_index=False)
         .agg(horas=('horas_evento', 'sum'), eventos=('cd_equipamento', 'size'))
         .sort_values('horas', ascending=False)
)
res_sem_maquina.head(20)


In [ ]:
fig = px.bar(
    res_sem_maquina.head(20),
    x='cd_equipamento',
    y='horas',
    color='eventos',
    title='Sem apontamento por maquina',
    labels={'cd_equipamento': 'Maquina', 'horas': 'Horas sem apontamento', 'eventos': 'Eventos'}
)
fig.update_layout(template='plotly_white', height=420)
fig.show()


In [ ]:
if {'latitude_interpolacao', 'longitude_interpolacao'}.issubset(df_sem.columns):
    mapa_sem = df_sem.dropna(subset=['latitude_interpolacao', 'longitude_interpolacao']).copy()
    mapa_sem['latitude_interpolacao'] = pd.to_numeric(mapa_sem['latitude_interpolacao'], errors='coerce')
    mapa_sem['longitude_interpolacao'] = pd.to_numeric(mapa_sem['longitude_interpolacao'], errors='coerce')
    mapa_sem = mapa_sem.dropna(subset=['latitude_interpolacao', 'longitude_interpolacao'])

    if not mapa_sem.empty:
        fig = px.density_mapbox(
            mapa_sem,
            lat='latitude_interpolacao',
            lon='longitude_interpolacao',
            z='horas_evento',
            radius=12,
            zoom=9,
            height=520,
            mapbox_style='carto-positron',
            title='Mapa de concentracao de sem apontamento (ponderado por horas)'
        )
        fig.show()
    else:
        print('Sem coordenadas validas para mapa de sem apontamento.')
else:
    print('Colunas de coordenada nao disponiveis para mapa.')


## 4) Picos horarios de paradas especificas (ex.: FALTA CAMINHAO)


In [ ]:
df_paradas = df[df['estado_processo'].eq('Parada')].copy()
df_paradas['desc_parada_norm'] = df_paradas['desc_parada'].str.upper().str.strip()

parada_alvo_norm = PARADA_ALVO.upper().strip()
base_alvo = df_paradas[df_paradas['desc_parada_norm'].str.contains(parada_alvo_norm, na=False)].copy()

print('Parada alvo:', PARADA_ALVO)
print('Registros encontrados:', len(base_alvo))


In [ ]:
if base_alvo.empty:
    print('Sem dados para a parada alvo no periodo.')
else:
    pico_hora = (
        base_alvo.groupby('hora', as_index=False)
                .agg(horas=('horas_evento', 'sum'), eventos=('hora', 'size'))
                .sort_values('hora')
    )

    fig = go.Figure()
    fig.add_bar(x=pico_hora['hora'], y=pico_hora['horas'], name='Horas', marker_color='#c1121f')
    fig.add_trace(go.Scatter(x=pico_hora['hora'], y=pico_hora['eventos'], mode='lines+markers', name='Eventos', yaxis='y2', line=dict(color='#1d4e89')))
    fig.update_layout(
        template='plotly_white',
        height=420,
        title=f'Picos horarios - {PARADA_ALVO}',
        xaxis_title='Hora do dia',
        yaxis=dict(title='Horas de parada'),
        yaxis2=dict(title='Qtd eventos', overlaying='y', side='right')
    )
    fig.show()


In [ ]:
# Heatmap dia x hora para parada alvo
if not base_alvo.empty:
    heat = (
        base_alvo.groupby(['data', 'hora'], as_index=False)
                .agg(horas=('horas_evento', 'sum'))
    )
    heat_pivot = heat.pivot(index='hora', columns='data', values='horas').fillna(0)

    fig = go.Figure(
        data=go.Heatmap(
            z=heat_pivot.to_numpy(),
            x=[str(c) for c in heat_pivot.columns],
            y=heat_pivot.index,
            colorscale='YlOrRd',
            colorbar=dict(title='Horas')
        )
    )
    fig.update_layout(
        template='plotly_white',
        height=460,
        title=f'Heatmap de picos horarios - {PARADA_ALVO}',
        xaxis_title='Data',
        yaxis_title='Hora do dia'
    )
    fig.show()


In [ ]:
# Top paradas por horario de pico (para olhar alem de FALTA CAMINHAO)
if not df_paradas.empty:
    top_picos = (
        df_paradas.groupby(['desc_parada_norm', 'hora'], as_index=False)
                 .agg(horas=('horas_evento', 'sum'))
    )
    idx = top_picos.groupby('desc_parada_norm')['horas'].idxmax()
    top_picos = top_picos.loc[idx].sort_values('horas', ascending=False)
    top_picos = top_picos.rename(columns={'hora': 'hora_pico', 'desc_parada_norm': 'parada'})
    top_picos.head(20)


## Proximos passos sugeridos

1. Validar unidade de `vl_consumo_instantaneo` (L/h vs outra unidade) para fechar o calculo de litros.
2. Definir oficialmente o patio (lista fixa de maquinas) para evitar distorcao por frota incompleta no periodo.
3. Adicionar meta por turno (inicio/fim esperado) e medir desvio por maquina e por dia.
4. Cruzar picos de `FALTA CAMINHAO` com localizacao e talhao para identificar gargalo logistico.
